# **Instituto de Informática – UFRGS**
## Disciplina de Aprendizado de Máquina
#### *Profa. Mariana Recamonde Mendoza (mrmendoza@inf.ufrgs.br)*
<br>

---

***Observação:*** *Este notebook é disponibilizado como material complementar às aulas e slides fornecidos pela professora. Os conceitos fundamentais são apresentados no material teórico; aqui, o objetivo é reforçar a compreensão e demonstrar aspectos práticos do uso de algoritmos e estratégias de avaliação em Aprendizado de Máquina.*

---


# Exercício Prático – KNN no dataset Credit-G (OpenML id=31)

Neste exercício iremos:

1️⃣ Trabalhar inicialmente apenas com variáveis numéricas

2️⃣ Demonstrar o impacto da diferença de escala

3️⃣ Aplicar normalização

4️⃣ Incluir variáveis categóricas com one-hot encoding

5️⃣ Analisar o efeito do aumento da dimensionalidade nas distâncias


## (1) Importando bibliotecas


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

## (2) Carregando, explorando e preparando o dataset Credit-g


Link para o dataset: https://www.openml.org/search?type=data&sort=runs&id=31


In [ ]:
credit = fetch_openml(name='credit-g', version=2, as_frame=True)

df = credit.frame
df.head()

### EDA (Exploratory Data Analysis)





In [ ]:
# ==============================
# Análise Exploratória Básica
# ==============================

import matplotlib.pyplot as plt

print("===== VISÃO GERAL DO DATASET =====")
print(f"Número de instâncias: {df.shape[0]}")
print(f"Número total de atributos (incluindo classe): {df.shape[1]}")
print()

# Separando tipos de atributos
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

print("===== TIPOS DE ATRIBUTOS =====")
print(f"Número de atributos numéricos: {len(numeric_cols)}")
print(f"Número de atributos categóricos: {len(categorical_cols)}")
print()

# ==============================
# Distribuição da classe
# ==============================
print("===== DISTRIBUIÇÃO DA CLASSE =====")
print(df['class'].value_counts())
print()

plt.figure()
df['class'].value_counts().plot(kind='bar')
plt.title("Distribuição da Classe")
plt.xlabel("Classe")
plt.ylabel("Frequência")
plt.show()

In [ ]:
# ==============================
# Estatísticas numéricas
# ==============================
print("===== ESTATÍSTICAS DOS ATRIBUTOS NUMÉRICOS =====")
display(df[numeric_cols].describe())

# ==============================
# Cálculo da amplitude (max - min)
# ==============================

print("===== AMPLITUDE DOS ATRIBUTOS NUMÉRICOS =====")

amplitude = df[numeric_cols].max() - df[numeric_cols].min()

amplitude_df = pd.DataFrame({
    'Mínimo': df[numeric_cols].min(),
    'Máximo': df[numeric_cols].max(),
    'Amplitude (Max - Min)': amplitude
})

# Ordenando da maior para a menor amplitude
amplitude_df = amplitude_df.sort_values(by='Amplitude (Max - Min)', ascending=False)

display(amplitude_df)

---
### >> ❓ Pergunta

Avalie os atributos numéricos.

*   Qual atributo possui maior amplitude?
*   Se um atributo tem amplitude maior que outro (por exemplo, 10 vezes maior) qual será seu peso relativo na distância?
*   Isso é desejável ou arbitrário?
---

In [ ]:
# ==============================
# Distribuição categórica (com gráficos)
# ==============================
print("===== DISTRIBUIÇÃO DOS ATRIBUTOS CATEGÓRICOS =====")

for col in categorical_cols:
    print(f"\nAtributo: {col}")
    print(df[col].value_counts())

    plt.figure()
    df[col].value_counts().plot(kind='bar')
    plt.title(f"Distribuição de {col}")
    plt.xlabel("Categoria")
    plt.ylabel("Frequência")
    plt.xticks(rotation=45)
    plt.show()

### Reservando um conjunto de teste independente

In [ ]:
# Separando X (entradas) e y (saídas) para se adequar às funções do scikit-learn.
X = df.drop(columns=['class'])
y = df['class']

# Faz uma divisão dos dados: reserva 20% e deixa fora do treinamento do modelo.
# Este conjunto simula novos dados (nunca antes vistos) e são chamados de Dados de Teste.
# O restante dos dados (80%) é usado para treinamento do modelo.
# Garantimos fazer UM ÚNICO split para todos os experimentos - garante comparações corretas e justas.
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Tamanho treino:", X_train_full.shape)
print("Tamanho teste:", X_test_full.shape)

## (3) Separando variáveis numéricas e treinando um modelo de KNN

Inicialmente usaremos apenas atributos numéricos para treinar um modelo de kNN.


In [ ]:
# Identificando colunas numéricas
numeric_cols = X_train_full.select_dtypes(include=['int64', 'float64']).columns

# Aplicando a seleção ao treino e teste
X_train_num = X_train_full[numeric_cols]
X_test_num  = X_test_full[numeric_cols]

X_train_num.head()

In [ ]:
# ==============================
# Treinando o modelo KNN (sem normalização)
# ==============================

# Criamos um objeto do classificador KNN.
# n_neighbors=3 significa que o modelo irá considerar os 3 vizinhos mais próximos
# para decidir a classe de uma nova instância.
#
# metric="euclidean" define que a distância utilizada será a distância euclidiana.
# Ou seja, o modelo irá medir proximidade no espaço geométrico dos atributos.

modelo = KNeighborsClassifier(n_neighbors=3, metric="euclidean")


# O método .fit() no KNN NÃO constrói um modelo paramétrico.
# Ele simplesmente armazena os dados de treino.
# No momento da predição, o algoritmo calculará a distância
# entre cada ponto de teste e TODOS os pontos de treino.
modelo.fit(X_train_num, y_train)


# O método .predict() realiza:
# 1) Cálculo das distâncias entre cada instância de teste e todas as instâncias de treino
# 2) Seleção dos k vizinhos mais próximos
# 3) Votação majoritária entre as classes desses vizinhos
y_pred = modelo.predict(X_test_num)


# Acurácia = Taxa total de acerto = número de predições corretas / total de instâncias analisadas
# Aqui estamos avaliando o desempenho no conjunto reservado previamente (conjunto de teste)
print('Acurácia sem normalização:', accuracy_score(y_test, y_pred))

## (4) Aplicando normalização (método MinMaxScaler)


In [ ]:
# ==============================
# Aplicando Normalização (Min-Max Scaling)
# ==============================

# Criamos o objeto responsável pela normalização.
# O MinMaxScaler transforma cada atributo individualmente
# para o intervalo [0, 1], usando a fórmula:
#
# X_norm = (X - min(X)) / (max(X) - min(X))
#
# Assim, o menor valor da coluna vira 0
# e o maior valor da coluna vira 1.
#
# Isso impede que atributos com grande amplitude
# dominem o cálculo da distância euclidiana (ou outras métricas sensíveis).

scaler = MinMaxScaler()


# IMPORTANTE:
# fit_transform() é aplicado APENAS no conjunto de treino.
#
# O método fit() calcula os valores de mínimo e máximo
# usando SOMENTE os dados de treino.
#
# O método transform() aplica essa mesma transformação.
# Como veremos mais adiante, isso evita vazamento de informação do conjunto de teste para o treino (data leakage).

X_train_num_norm = scaler.fit_transform(X_train_num)


# No conjunto de teste usamos APENAS transform().
# Ou seja, aplicamos os mesmos min e max aprendidos no treino.
# Nunca devemos recalcular a normalização usando dados de teste.

X_test_num_norm  = scaler.transform(X_test_num)


# Agora treinamos novamente o KNN,
# mas no espaço transformado (normalizado).

modelo.fit(X_train_num_norm, y_train)


# A predição agora é feita considerando
# distâncias calculadas no novo espaço [0,1]^d.

y_pred = modelo.predict(X_test_num_norm)


# Avaliamos novamente a acurácia no conjunto de teste.
print('Acurácia com normalização:', accuracy_score(y_test, y_pred))

---
### >> ❓ Pergunta


*  A normalização alterou o desempenho neste experimento? Em que sentido?
*  Por que o KNN é particularmente sensível à escala?
---

### 📌 Observação Importante sobre Normalização

A normalização nem sempre produzirá o mesmo tipo de alteração no desempenho do modelo.

Em alguns conjuntos de dados, o impacto pode ser significativo.
Em outros, pode ser pequeno ou até imperceptível.

Isso ocorre porque o efeito da normalização depende:

- Da diferença de escala entre os atributos  
- Da distribuição dos valores  
- Da geometria do espaço de dados  
- Do próprio algoritmo utilizado  

⚠️ O ponto central não é “sempre melhorar o desempenho do modelo”.

O objetivo da normalização é garantir que **nenhum atributo receba um peso maior na decisão do modelo apenas por estar em uma escala numérica maior**.

Sem normalização, atributos com maior amplitude podem dominar o cálculo da distância euclidiana, introduzindo uma ponderação implícita e não intencional.

Ao normalizar, estamos garantindo que todos os atributos contribuam de forma comparável para o cálculo das distâncias — deixando que o modelo aprenda padrões reais, e não diferenças artificiais de escala.

## (5) Comparando valores de k

No algoritmo k-Nearest Neighbors (KNN), o valor de **k** representa o número de vizinhos mais próximos considerados para decidir a classe de uma nova instância.

Esse valor é um exemplo de **hiperparâmetro**.

### 📌 O que é um hiperparâmetro?

Em Aprendizado de Máquina, hiperparâmetros **são configurações definidas antes do treinamento do modelo**.  
Eles não são aprendidos a partir dos dados, mas sim escolhidos pelo usuário.

Exemplos:
- O valor de *k* no KNN  
- A profundidade máxima de uma árvore de decisão
- A taxa de aprendizado em redes neurais  

No KNN, o valor de k controla o** nível de suavização** da decisão:

- k pequeno → decisão baseada em poucos vizinhos → modelo mais sensível a ruído  
- k grande → decisão baseada em muitos vizinhos → modelo mais estável, porém mais “suavizado”  

Nesta seção, iremos comparar diferentes valores de k utilizando:

- Apenas variáveis numéricas  
- Distância euclidiana  
- O mesmo conjunto de treino e teste  

O objetivo é observar como a escolha do hiperparâmetro influencia o desempenho do modelo.

In [ ]:
# =====================================================
# Comparação de diferentes valores de k (dados numéricos)
# =====================================================

# Nesta análise:
# - Usamos APENAS as variáveis numéricas (X_train_num e X_test_num)
# - Mantemos o MESMO conjunto para treinar e avaliar o modelo
# - Utilizamos distância euclidiana
# - Avaliamos desempenho via taxa de acerto (chamada de acurácia)
#
# O objetivo é observar como o hiperparâmetro k influencia a decisão do modelo.

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Valores de k a serem comparados
valores_k = [1, 3, 5, 7, 9, 11]

# Lista para armazenar as acurácias
acuracias = []

for k in valores_k:

    # Criamos o modelo KNN com:
    # - n_neighbors = k
    # - metric = "euclidean" (distância euclidiana)
    modelo = KNeighborsClassifier(n_neighbors=k, metric="euclidean")

    # Treinamos o modelo usando apenas dados numéricos
    modelo.fit(X_train_num, y_train)

    # Realizamos a predição no conjunto de teste
    y_pred = modelo.predict(X_test_num)

    # Calculamos a acurácia
    acc = accuracy_score(y_test, y_pred)

    # Armazenamos o resultado
    acuracias.append(acc)

    print(f"k = {k} | Acurácia = {acc:.4f}")


# =====================================================
# Visualização dos resultados
# =====================================================

plt.figure()

bars = plt.bar([str(k) for k in valores_k], acuracias)

plt.xlabel("Valor de k")
plt.ylabel("Acurácia")
plt.title("Comparação de Acurácia para Diferentes Valores de k\n(Dados Numéricos, Distância Euclidiana)")

# Inserindo o valor da acurácia em cima de cada barra
for bar, acc in zip(bars, acuracias):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2,
             height,
             f"{acc:.3f}",
             ha='center',
             va='bottom')

plt.show()

---
### >> ❓ Pergunta


*  Qual foi o melhor valor de k neste experimento?
---

In [ ]:
# Configure aqui o melhor valor de k.
# Por padrão, estamos usando 1 (para evitar erros de execução no código por falta de inicialização)
best_k = 1

## (6) Adicionando variáveis categóricas (One-Hot Encoding)


Até agora utilizamos apenas variáveis numéricas.

No entanto, o dataset credit-g contém diversos atributos categóricos.  
Como o KNN calcula distâncias numéricas, precisamos transformar essas variáveis em uma representação numérica adequada. Isso é uma tarefa de pré-processamento de dados, e se chama de **codificação de atributos.**

A técnica mais comum para isso é o **one-hot encoding**, que cria uma nova dimensão para cada categoria possível de um atributo. Ela é adequada para atributos categóricos nominais (quando não existe uma relação de ordem explícita).

### 📌 Impacto na dimensionalidade

Se um atributo possui 4 categorias, o one-hot encoding cria 4 novas colunas.
Se vários atributos categóricos estiverem presentes, o número total de dimensões pode crescer rapidamente.

Esse aumento na dimensionalidade altera:

- A geometria do espaço de dados  
- O cálculo das distâncias  
- O comportamento do KNN  

Na próxima etapa, vamos aplicar o one-hot encoding e observar como a dimensionalidade do problema se modifica.

In [ ]:
# One-hot encoding no conjunto de treino
X_train_encoded = pd.get_dummies(X_train_full, drop_first=False)

# Aplicando as mesmas colunas ao teste
X_test_encoded = pd.get_dummies(X_test_full, drop_first=False)

# Garantindo mesmas colunas (alinhamento)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print("Dimensionalidade após encoding:", X_train_encoded.shape[1])


In [ ]:
X_test_encoded


Agora treinamos novamente o modelo, com os mesmos hiperparâmetros definidos anteriormente, averiguando o desempenho

In [ ]:
# Aplicando a normalização
scaler = MinMaxScaler()

X_train_all = scaler.fit_transform(X_train_encoded)
X_test_all  = scaler.transform(X_test_encoded)

# Treinando o modelo de KNN.
# Usamos o melhor valor de k encontrado na etapa (5)
modelo = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean")

modelo.fit(X_train_all, y_train)
y_pred = modelo.predict(X_test_all)

print('Acurácia com todas as variáveis:', accuracy_score(y_test, y_pred))

---
### >> ❓ Pergunta


*  Qual foi o impacto da inclusão das variáveis categóricas no desempenho do modelo?
---

## (7) Análise do efeito da dimensionalidade nas distâncias

Vamos observar a diferença entre a menor e a maior distância de um ponto.


In [ ]:
# =====================================================
# Efeito da dimensionalidade na distribuição das distâncias
# =====================================================

from sklearn.metrics import pairwise_distances
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------------------------------
# 1) Espaço com APENAS variáveis numéricas (normalizadas)
# -----------------------------------------------------

dist_num = pairwise_distances(X_train_num_norm)

# Removendo a diagonal (distância zero consigo mesmo)
dist_num = dist_num[np.nonzero(dist_num)]

min_num = np.min(dist_num)
max_num = np.max(dist_num)
contraste_num = (max_num - min_num) / min_num

print("===== Espaço Numérico =====")
print("Dimensionalidade:", X_train_num_norm.shape[1])
print("Menor distância:", round(min_num, 4))
print("Maior distância:", round(max_num, 4))
print("Contraste (max - min)/min:", round(contraste_num, 4))
print()

# Histograma
plt.figure()
plt.hist(dist_num, bins=30)
plt.title("Distribuição das Distâncias\n(Apenas Variáveis Numéricas)")
plt.xlabel("Distância Euclidiana")
plt.ylabel("Frequência")
plt.show()


# -----------------------------------------------------
# 2) Espaço com variáveis numéricas + categóricas
# -----------------------------------------------------

dist_all = pairwise_distances(X_train_all)
dist_all = dist_all[np.nonzero(dist_all)]

min_all = np.min(dist_all)
max_all = np.max(dist_all)
contraste_all = (max_all - min_all) / min_all

print("===== Espaço Numérico + Categórico =====")
print("Dimensionalidade:", X_train_all.shape[1])
print("Menor distância:", round(min_all, 4))
print("Maior distância:", round(max_all, 4))
print("Contraste (max - min)/min:", round(contraste_all, 4))
print()

# Histograma
plt.figure()
plt.hist(dist_all, bins=30)
plt.title("Distribuição das Distâncias\n(Numérico + One-Hot Encoding)")
plt.xlabel("Distância Euclidiana")
plt.ylabel("Frequência")
plt.show()


# -----------------------------------------------------
# Comparação Final
# -----------------------------------------------------

print("===== Comparação =====")
print("Contraste espaço numérico: ", round(contraste_num, 4))
print("Contraste espaço ampliado: ", round(contraste_all, 4))

if contraste_all < contraste_num:
    print("\n➡ As distâncias estão mais concentradas no espaço de maior dimensionalidade.")
else:
    print("\n➡ O efeito depende da estrutura dos dados.")

---

### >> ❓ Pergunta

Com base nos experimentos realizados comparando:

- O espaço com apenas variáveis numéricas  
- O espaço com variáveis numéricas + categóricas (one-hot encoding)  

Responda:

*  O que aconteceu com a distribuição das distâncias quando a dimensionalidade aumentou?
*  As distâncias ficam mais concentradas?
*  Como isso pode impactar o desempenho do KNN?

---

---

## 📌 Conectando com a Maldição da Dimensionalidade

Nos experimentos realizados, observamos que, ao aumentar a dimensionalidade do espaço (especialmente após aplicar one-hot encoding), ocorreu:

- Aumento no número de atributos  
- Concentração das distâncias  
- Redução da diferença relativa entre o vizinho mais próximo e o mais distante  

Esse fenômeno está relacionado ao que chamamos de **Maldição da Dimensionalidade**.

### O que significa isso?

À medida que a dimensionalidade cresce:

- O espaço se torna progressivamente mais “vazio”  
- As distâncias entre os pontos tendem a se tornar mais semelhantes  
- O contraste entre pontos próximos e distantes diminui  

Para algoritmos baseados em distância, como o KNN, isso pode ser problemático, porque:

- O vizinho mais próximo deixa de ser significativamente mais próximo  
- A noção de “proximidade” perde poder discriminativo  
- O modelo pode ter mais dificuldade para distinguir padrões  

### 🎯 Conclusão

Mais atributos não significam necessariamente mais informação útil.

Em muitos casos, o aumento da dimensionalidade pode:

- Introduzir ruído  
- Reduzir o contraste geométrico entre instâncias  
- Impactar negativamente algoritmos baseados em distância  

Por isso, em problemas de alta dimensionalidade, técnicas como:

- Seleção de atributos  
- Redução de dimensionalidade  
- Engenharia cuidadosa de variáveis  

podem ser essenciais para manter o desempenho do modelo.

---